# Pipeline

In [6]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight 
# Carga tu CSV con los datos de entrenamiento
df_train = pd.read_csv("../Datos/Original/tcga_simple_train.csv")

# Nombre exacto de las columnas
columna_texto = 'text' 
columna_etiqueta = 't' 

X_train_raw = df_train[columna_texto].astype(str)
y_train_raw = df_train[columna_etiqueta]


# Convertimos las etiquetas (ej. 'T1', 'T2') a números (0, 1, 2, 3)
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)

# Parámetros del vocabulario
VOCAB_SIZE = 5000
MAX_LEN = 300

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_raw)

# Convertimos texto a secuencias numéricas y aplicamos padding
X_train_seq = tokenizer.texts_to_sequences(X_train_raw)
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')

# Balanceamos las clases para que el modelo aprenda bien a detectar el estadio T4
pesos_clases = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
dict_pesos = dict(enumerate(pesos_clases))


num_clases = len(label_encoder.classes_)

model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=100, input_length=MAX_LEN),
    Conv1D(filters=256, kernel_size=10, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(128, activation='relu'),
    Dropout(0.75),
    Dense(num_clases, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Le pasamos el dict_pesos al entrenamiento para que castigue los fallos en T1 y T4
model.fit(X_train_pad, y_train, epochs=10, batch_size=32, validation_split=0.1, class_weight=dict_pesos, verbose= 0)


df_nuevo = pd.read_csv("../Datos/A_Predecir/tcga_simple_test_empty.csv")

X_nuevo_raw = df_nuevo[columna_texto].astype(str)

X_nuevo_seq = tokenizer.texts_to_sequences(X_nuevo_raw)
X_nuevo_pad = pad_sequences(X_nuevo_seq, maxlen=MAX_LEN, padding='post', truncating='post')

predicciones_prob = model.predict(X_nuevo_pad)

predicciones_num = np.argmax(predicciones_prob, axis=1)

predicciones_etiquetas = label_encoder.inverse_transform(predicciones_num)

df_nuevo[columna_etiqueta] = predicciones_etiquetas

df_nuevo.to_csv("../Datos/A_Predecir/tcga_simple_test_empty.csv", index=False)

c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
